# Fine-tune your own Gemma 4 (E2B) / 自分だけの Gemma 4 (E2B) を育てる

This free Colab notebook teaches a small AI to talk **your way**, using the
examples you prepared. It runs on a **free T4 GPU** — no paid plan needed.

この無料 Colab ノートブックは、あなたが用意した例を使って小さな AI に
**あなたらしい話し方**を教えます。**無料の T4 GPU** で動きます。

**Before you start / はじめる前に**
1. Runtime ▸ Change runtime type ▸ **T4 GPU** を選択 / select **T4 GPU**.
2. Make your data with `prepare_data.py` and keep `out/train_sharegpt.jsonl` ready.
   `prepare_data.py` でデータを作り、`out/train_sharegpt.jsonl` を用意。
3. Runtime ▸ **Run all** / すべて実行。

Steps: **install → load Gemma 4 → add LoRA → your data → train → test → export for Ollama**.
手順: **準備 → Gemma 4 を読込 → LoRA 追加 → データ → 学習 → テスト → Ollama 用に書き出し**。

## 1. Install Unsloth / Unsloth をインストール
Takes 1-2 minutes. / 1〜2分かかります。

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

## 2. Load Gemma 4 E2B / Gemma 4 E2B を読み込む
We use the instruct model, the same one you can run in Ollama.
Ollama でも動かせる instruct モデルを使います。

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None,            # auto
    max_seq_length = 1024,   # keep examples shorter than this
    load_in_4bit = False,
    full_finetuning = False,
)

## 3. Add LoRA adapters / LoRA を追加
LoRA trains a tiny set of extra weights (~0.2%) instead of the whole
model — fast, and small to share.
LoRA はモデル全体ではなく、ごく小さな追加の重み(約0.2%)だけを学習します。

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # text only
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 8, lora_alpha = 8, lora_dropout = 0,
    bias = "none", random_state = 3407,
)

## 4. Load YOUR data / あなたのデータを読み込む
Your data can come from **three** places. Set `DATA_SOURCE` in the next
cell, then run.
データの取り込み方は **3通り**。次のセルの `DATA_SOURCE` を選んで実行します。

- `"upload"` — pick a file from your computer (a CSV or a .jsonl)
  パソコンからファイルを選ぶ(CSV か .jsonl)
- `"github"` — load your own file from **your** GitHub (paste its Raw URL)
  **自分の** GitHub に置いたファイルを読む(Raw の URL を貼る)
- `"example"` — try a ready-made example dataset / 用意された例で試す

A **CSV** is auto-cleaned + converted by `prepare_data.py`; a **.jsonl**
(already prepared) is used as-is.
**CSV** は `prepare_data.py` が自動で整形・変換、**.jsonl** はそのまま使います。

In [ ]:
# Choose ONE / どれか1つ:  "upload" | "github" | "example"
DATA_SOURCE = "upload"

# For "github" only: paste your file's RAW url
# (open the file on GitHub, click the "Raw" button, copy that URL).
# 「github」のときだけ: GitHub でファイルを開き "Raw" を押した URL を貼る。
MY_DATA_URL = ""  # e.g. https://raw.githubusercontent.com/you/your-repo/main/mydata.csv

In [ ]:
import urllib.request, os
LAB_RAW = "https://raw.githubusercontent.com/itoksk/gemma-finetune-lab/main"

if DATA_SOURCE == "upload":
    from google.colab import files
    print("Choose your file (CSV or .jsonl) / ファイルを選ぶ")
    up = files.upload(); src = next(iter(up))
elif DATA_SOURCE == "github":
    assert MY_DATA_URL, "Set MY_DATA_URL first / さきに MY_DATA_URL を設定"
    src = MY_DATA_URL.split("/")[-1].split("?")[0]
    urllib.request.urlretrieve(MY_DATA_URL, src)
else:  # example
    urllib.request.urlretrieve(LAB_RAW + "/data/examples/cafe_faq.csv", "cafe_faq.csv")
    src = "cafe_faq.csv"
print("Got:", src)

# A CSV is cleaned + converted by our data tool; a .jsonl is ready already.
# CSV は整形ツールで変換、.jsonl はそのまま。
if src.endswith(".csv"):
    urllib.request.urlretrieve(LAB_RAW + "/scripts/prepare_data.py", "prepare_data.py")
    os.system(f"python3 prepare_data.py {src}")
    data_file = "out/train_sharegpt.jsonl"
else:
    data_file = src
print("data_file =", data_file)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("json", data_files=data_file, split="train")
print(dataset)
print(dataset[0])

Apply the Gemma 4 chat template and turn each conversation into
one training string.
Gemma 4 のチャットテンプレートを当て、各会話を1つの学習用テキストにします。

In [ ]:
from unsloth.chat_templates import get_chat_template, standardize_data_formats
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-4")
dataset = standardize_data_formats(dataset)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix('<bos>') for c in convos]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset[0]["text"])

## 5. Train / 学習する
For a small custom dataset, training for a few **epochs** (full passes over
your data) works better than a fixed step count. 2-4 is a good range.
小さな自作データでは、固定ステップより数 **エポック**(データを丸ごと何周
するか)の方が良い結果になります。2〜4 が目安です。

In [ ]:
EPOCHS = 3   # try 2-4 / 2〜4 で試す

from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer,
    train_dataset = dataset, eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = EPOCHS,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

Train only on the assistant's replies (not the user's questions) —
this makes the fine-tune sharper.
ユーザーの質問ではなく AI の返答だけを学習対象にします。精度が上がります。

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

In [ ]:
trainer_stats = trainer.train()
print("Done! / 完了!", trainer_stats.metrics.get("train_runtime"), "seconds")

## 6. Test it / 試す
Change the question and re-run to chat with your fine-tune.
質問を書き換えて再実行すると、育てた AI と会話できます。

In [ ]:
from transformers import TextStreamer
messages = [{
    "role": "user",
    "content": [{"type": "text", "text": "What time do you open?"}]
}]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors='pt',
    tokenize=True, return_dict=True,
).to("cuda")
_ = model.generate(**inputs, max_new_tokens=128,
                   temperature=1.0, top_p=0.95, top_k=64,
                   streamer=TextStreamer(tokenizer, skip_prompt=True))

## 7. Export for Ollama / Ollama 用に書き出す
Two ways to take your model home — run the one you want.
持ち帰る方法は2つ。使いたい方を実行してください。

- **A. Merged GGUF (recommended / おすすめ)** — one self-contained file that
  runs in Ollama on any computer.
- **B. LoRA adapter (small / 軽い)** — a tiny folder that sits on top of the
  `gemma4:e2b` you already have in Ollama.

### A. Merged GGUF / 統合した GGUF
`q4_k_m` is small and fits the free tier. Use `q8_0` for higher
quality if it doesn't run out of memory.
`q4_k_m` は小さく無料枠に収まります。余裕があれば `q8_0` で高品質に。

In [ ]:
model.save_pretrained_gguf(
    "gemma_ft_gguf", tokenizer,
    quantization_method = "q4_k_m",   # or "q8_0"
)

# Find the .gguf and give it a simple name / .gguf を探して分かりやすい名前に
import glob, shutil, os
g = sorted(glob.glob("gemma_ft_gguf/*.gguf"))
assert g, "No GGUF produced"
shutil.copy(g[0], "my-gemma.gguf")
print("GGUF:", os.path.getsize("my-gemma.gguf")//1_000_000, "MB")

In [ ]:
# Write a Modelfile next to the GGUF / GGUF と一緒に Modelfile を作る
modelfile = '''FROM ./my-gemma.gguf
PARAMETER temperature 1.0
PARAMETER top_p 0.95
PARAMETER top_k 64
'''
open("Modelfile", "w").write(modelfile)

from google.colab import files
files.download("my-gemma.gguf")
files.download("Modelfile")

### B. LoRA adapter only / LoRA アダプターだけ
Tiny. On your computer, point Ollama at it with `FROM gemma4:e2b` +
`ADAPTER`. See `use/Modelfile.example`.
とても軽量。手元では `FROM gemma4:e2b` + `ADAPTER` で使います。

In [ ]:
model.save_pretrained("gemma_lora")
tokenizer.save_pretrained("gemma_lora")
!cd gemma_lora && zip -qr ../gemma_lora.zip . && cd ..
from google.colab import files
files.download("gemma_lora.zip")

## 8. Run it at home / 自宅で動かす
On your own computer (Ollama installed):
自分のパソコンで(Ollama を入れた状態で):

```bash
# A) merged GGUF — put my-gemma.gguf and Modelfile in one folder, then:
ollama create my-gemma -f Modelfile
ollama run my-gemma "What time do you open?"
```

Then **chat in a browser** (open `use/web-chat/index.html`) or **use it in
vibe-local** (`use/vibe-local.md`). Full guide: the repo `README.md`.
そのあと **ブラウザで会話**(`use/web-chat/index.html`)したり、**vibe-local で
使ったり**(`use/vibe-local.md`)できます。詳しくは `README.md`。

> If answers seem off, re-export at `q8_0` (step 7A) and make sure your
> Modelfile matches. Gemma 4 E2B is brand new, so quality can vary by tool.
> 返答が変なときは `q8_0` で書き出し直し、Modelfile を確認してください。